# Reconnaissance vocale avec RNN/LSTM et CTC
Notebook pédagogique step-by-step en PyTorch

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torchaudio.transforms import MFCC
from torch.utils.data import DataLoader, Dataset


In [ ]:
class AudioDataset(Dataset):
    def __init__(self, audio_paths, transcripts, sample_rate=16000, n_mfcc=13):
        self.audio_paths = audio_paths
        self.transcripts = transcripts
        self.sample_rate = sample_rate
        self.mfcc_transform = MFCC(sample_rate=sample_rate, n_mfcc=n_mfcc)
        self.char_to_idx = {c:i for i,c in enumerate("ABCDEFGHIJKLMNOPQRSTUVWXYZ '")}
        self.blank_idx = len(self.char_to_idx)

    def __len__(self):
        return len(self.audio_paths)

    def __getitem__(self, idx):
        waveform, sr = torchaudio.load(self.audio_paths[idx])
        mfcc = self.mfcc_transform(waveform).squeeze(0).transpose(0,1)
        targets = [self.char_to_idx[c] for c in self.transcripts[idx] if c in self.char_to_idx]
        return mfcc, torch.tensor(targets)


In [7]:
class SpeechRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim*2, output_dim)

    def forward(self, x):
        out,_ = self.lstm(x)
        return self.fc(out).log_softmax(2)
